<a href="https://colab.research.google.com/github/beniamine3155/Data_Science_Projects/blob/main/Skin_Disease_Diagnosis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Importing necessary Libraries

In [ ]:
# set seeds for reproducibility
import random
random.seed(0)

import numpy as np
np.random.seed(0)

import tensorflow as tf
tf.random.set_seed(0)

In [7]:
import os
import json
from zipfile import ZipFile
from PIL import Image

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.image as mpimg
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
from tensorflow.keras.models import Model


### Data Curation

In [ ]:
!pip install kaggle

In [8]:
kaggle_credentials = json.load(open('kaggle.json'))

In [9]:
# setup kaggle API key as environmen variables
os.environ['KAGGLE_USERNAME'] = kaggle_credentials['username']
os.environ['KAGGLE_KEY'] = kaggle_credentials['key']

In [13]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [14]:
!kaggle datasets list


ref                                                        title                                        size  lastUpdated          downloadCount  voteCount  usabilityRating  
---------------------------------------------------------  ------------------------------------------  -----  -------------------  -------------  ---------  ---------------  
bhadramohit/customer-shopping-latest-trends-dataset        Customer Shopping (Latest Trends) Dataset    76KB  2024-11-23 15:26:12          10306        202  1.0              
zafarali27/netflix-movies-and-tv-shows                     Netflix Movies and TV Shows                  28KB  2024-11-23 07:53:10           2489         48  1.0              
hopesb/student-depression-dataset                          Student Depression Dataset.                 454KB  2024-11-22 17:56:03           6798         92  0.9411765        
mujtabamatin/air-quality-and-pollution-assessment          Air Quality and Pollution Assessment         84KB  2024-12-04 15:2

In [15]:
!kaggle datasets download -d pacificrm/skindiseasedataset --unzip


Dataset URL: https://www.kaggle.com/datasets/pacificrm/skindiseasedataset
License(s): CC0-1.0
100% 1.36G/1.36G [01:14<00:00, 21.8MB/s]
100% 1.36G/1.36G [01:14<00:00, 19.7MB/s]


In [16]:
!ls

kaggle.json  Readme.md	sample_data  SkinDisease


In [ ]:
# unzip the downloaded dataset
# with ZipFile('plantvillage-dataset.zip', 'r') as zip_ref:
#   zip_ref.extractall()

### Step 1: Load the Data

In [17]:
import tensorflow as tf

train_dir = '/content/SkinDisease/SkinDisease/train'
test_dir = '/content/SkinDisease/SkinDisease/test'

# Load training dataset
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels="inferred",
    label_mode="int",
    image_size=(256, 256),
    batch_size=32,
    shuffle=True
)

# Load testing dataset
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels="inferred",
    label_mode="int",
    image_size=(256, 256),
    batch_size=32,
    shuffle=False
)

# Split training dataset into train and validation datasets
val_split = 0.2
train_dataset_size = len(train_dataset)
val_size = int(train_dataset_size * val_split)

train_dataset = train_dataset.skip(val_size)
val_dataset = train_dataset.take(val_size)


Found 13898 files belonging to 22 classes.
Found 1546 files belonging to 22 classes.


### Step 2: Normalize and Augment Data

In [18]:
# Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2)
])

# Normalize pixel values to [0, 1]
def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Apply normalization and augmentation
train_dataset = train_dataset.map(normalize).map(lambda x, y: (data_augmentation(x), y))
val_dataset = val_dataset.map(normalize)
test_dataset = test_dataset.map(normalize)


In [19]:
train_dataset

<_MapDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [20]:
val_dataset

<_MapDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [21]:
test_dataset

<_MapDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

### Step 3: Train a CNN Model

In [23]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Get the number of classes from the original training directory
num_classes = len(os.listdir(train_dir))  # Assuming each subdirectory is a class

# Define CNN model
cnn_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')  # Output layer using num_classes
])

# Compile the model
cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Train the model
history = cnn_model.fit(train_dataset, validation_data=val_dataset, epochs=10)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 246s 648ms/step - accuracy: 0.1504 - loss: 2.8917 - val_accuracy: 0.1986 - val_loss: 2.6946
Epoch 2/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 226s 619ms/step - accuracy: 0.2021 - loss: 2.6926 - val_accuracy: 0.2144 - val_loss: 2.6024
Epoch 3/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 225s 623ms/step - accuracy: 0.2164 - loss: 2.6246 - val_accuracy: 0.2507 - val_loss: 2.5236
Epoch 4/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 224s 613ms/step - accuracy: 0.2312 - loss: 2.5548 - val_accuracy: 0.2486 - val_loss: 2.4755
Epoch 5/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 225s 617ms/step - accuracy: 0.2420 - loss: 2.5221 - val_accuracy: 0.2644 - val_loss: 2.4391
Epoch 6/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 226s 621ms/step - accuracy: 0.2544 - loss: 2.4668 - val_accuracy: 0.2805 - val_loss: 2.3811
Epoch 7/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 225s 622ms/step - accuracy: 0.2598 - loss: 2.4197 - val_accuracy: 0.2698 - val_loss: 2.4050
Epoch 8/10
348/348 ━━━━━━━━━━━━━━━━━━━━ 260s 612ms/step - accuracy: 0.2676 -